In [ ]:
import pyEDM
import pandas as pd
import numpy as np
import random
from collections import Counter

p22 = pd.read_csv('../data/processed_22.csv')
p23 = pd.read_csv('../data/processed_23.csv')
p24 = pd.read_csv('../data/processed_24.csv')
p25 = pd.read_csv('../data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

In [ ]:
# top 20 features from paper + target variable
features = [
    "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
    "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
    "Solidity", "texture_uniformity", "texture_smoothness",
    "texture_average_gray_level", "texture_entropy",
    "texture_average_contrast", "H90", "H180", "Hflip",
    "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
]

df = combined[["date"] + features].copy()
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
df = df.sort_values("date").set_index("date")
df = df.asfreq("D")

df_filled = df.copy()

for col in features:
    ema = df[col].ewm(span=30, adjust=False).mean()
    df_filled[col] = df[col].fillna(ema)

# normalize features
df_norm = df_filled.copy()
for col in features:
    mu = df_filled[col].mean()
    sigma = df_filled[col].std()
    df_norm[col] = (df_filled[col] - mu) / sigma

df_mv = df_norm.copy()
df_mv = df_mv.reset_index()
df_mv["t"] = np.arange(1, len(df_mv) + 1)
df_mv = df_mv[["t"] + features]

target = "Lpoly_expected_ml"
predictors = [col for col in features if col != target]

df_mv.head()

In [ ]:
env = pd.read_csv("../data/environment_all.csv")

# could be relevant but too many missing values for ema imputation to be effective
env = env.drop(columns=[
    'fluorescent_dissolved_organic_matter_eco',
    'sea_water_turbidity_eco',
    'waterlevel_predicted_m',
    'mass_concentration_of_oxygen_in_sea_water_seaphox',
    'mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox',
    'fractional_saturation_of_oxygen_in_sea_water_seaphox',
    'sea_water_ph_reported_on_total_scale_seaphox_external'
])

env["date"] = pd.to_datetime(env["date"].astype(str), format="%Y%m%d")
env = env.sort_values("date").set_index("date")
env = env.asfreq("D")

env_features = env.columns.tolist()
env_filled = env.copy()

for col in env_features:
    ema = env[col].ewm(span=30, adjust=False).mean()
    env_filled[col] = env[col].fillna(ema)

df_norm = env_filled.copy()

for col in env_features:
    mu = env_filled[col].mean()
    sigma = env_filled[col].std()
    df_norm[col] = (env_filled[col] - mu) / sigma

df_env = df_norm.copy()
df_env = df_env.reset_index()
df_env["t"] = np.arange(1, len(df_env) + 1)
df_env = df_env[["t"] + env_features]

df_all = pd.merge(df_mv, df_env, on="t", how="inner")

df_env = df_env.merge(df_mv[["t", target]], on="t", how="inner")
df_env.head()

In [ ]:
df_all.head()

https://sugiharalab.github.io/EDM_Documentation/parameters/

In [ ]:
def one_simplex(df, target, features, E=4, Tp=1):
    # Randomly select 3 features (+ the target) for the simplex projection
    chosen_features = random.sample(features, 3)
    columns = [target] + chosen_features
    columns_str = " ".join(columns) # has to be 'space separated' idk ????

    N = len(df)
    res = pyEDM.Simplex(
        dataFrame=df,
        columns=columns_str,
        target=target,
        E=E,
        tau=1,
        Tp=Tp,
        lib=f"1 {N}",
        pred=f"1 {N}"
    )

    obs = res["Observations"].to_numpy()
    pred = res["Predictions"].to_numpy()

    mask = np.isfinite(obs) & np.isfinite(pred)
    obs = obs[mask]
    pred = pred[mask]

    if len(obs) < 10 or np.std(obs) == 0 or np.std(pred) == 0:
        return np.nan, chosen_features

    rho = np.corrcoef(obs, pred)[0, 1]
    rmse = np.sqrt(np.mean((obs - pred) ** 2))
    mae = np.mean(np.abs(obs - pred))
    return rho, rmse, mae, chosen_features

def multiview_big(df, target, features, Tp, n_trials=500):
    results = []

    for i in range(n_trials):
        rho, rmse, mae, chosen = one_simplex(df, target, features, E=4, Tp=Tp)

        results.append({
            "rho": rho,
            "rmse": rmse,
            "mae": mae,
            "features": chosen
        })

    return pd.DataFrame(results)

def multiview_yes(df_mv, target, predictors):
    # wrapper main function
    x = df_mv[target].to_numpy()

    summary_rows = []
    feature_importance_by_tp = {}
    
    for Tp in range(1, 32):
        mv = multiview_big(df_mv, target, predictors, Tp, n_trials=500)

        acf = abs(pd.Series(x).autocorr(lag=Tp))
        mv["rho_eff"] = mv["rho"] - acf

        # summary stats
        summary_rows.append({
            "Tp": Tp,
            "rho_eff_mean": mv["rho_eff"].mean(),
            "rmse_mean": mv["rmse"].mean(),
            "mae_mean": mv["mae"].mean(),
            "n_models": len(mv)
        })

        # feature importance (top 5%)
        top = mv.nlargest(int(0.05 * len(mv)), "rho_eff")

        counts = Counter()
        for feats in top["features"]:
            for f in feats:
                counts[f] += 1

        importance = pd.Series(counts) / len(top)
        feature_importance_by_tp[Tp] = importance.sort_values(ascending=False)
    
    return summary_rows, feature_importance_by_tp

In [ ]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_mv, target, predictors)
summary_df = pd.DataFrame(summary_rows)
# print(summary_df)

In [ ]:
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", None)

importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
# print(importance_all)

In [ ]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_env, target, env_features)
summary_df = pd.DataFrame(summary_rows)
# print(summary_df)

In [ ]:
importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
# print(importance_all)

In [ ]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_all, target, predictors+env_features)
summary_df = pd.DataFrame(summary_rows)
# print(summary_df)